In [1]:
import pandas as pd
import numpy as np

from traffic_monitoring_ml.utils import load_pickle, load_json
from traffic_monitoring_ml.config import COLLISION_VALUE_MAP_PATH, CASUALTY_VALUE_MAP_PATH, VEHICLE_VALUE_MAP_PATH, X_TRAIN_PATH, Y_TRAIN_PATH, GROUPS_TRAIN_PATH, SELECTED_FEATURES_PATH, X_TEST_PATH, Y_TEST_PATH, CATBOOST_MODEL_PATH

from pathlib import Path
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import optuna
from optuna.samplers import TPESampler


In [2]:
casualty_value_map = load_pickle(Path(CASUALTY_VALUE_MAP_PATH))
vehicle_value_map = load_pickle(Path(VEHICLE_VALUE_MAP_PATH))
collision_value_map = load_pickle(Path(COLLISION_VALUE_MAP_PATH))

In [3]:
X_train = (pd.read_parquet(Path(X_TRAIN_PATH))
           .replace(casualty_value_map)
           .replace(collision_value_map)
           .replace(vehicle_value_map))
X_test = (pd.read_parquet(Path(X_TEST_PATH))
           .replace(casualty_value_map)
           .replace(collision_value_map)
           .replace(vehicle_value_map))

y_train = pd.read_parquet(Path(Y_TRAIN_PATH)).squeeze()
y_test = pd.read_parquet(Path(Y_TEST_PATH)).squeeze()

In [4]:
selected_features = load_json(Path(SELECTED_FEATURES_PATH))
len(selected_features)

32

In [5]:
X_train = X_train[selected_features]
X_test = X_test[selected_features]

In [6]:
print(X_train.shape, X_test.shape)

(512334, 32) (128070, 32)


### Baseline CV

In [12]:
gkf = GroupKFold(n_splits=5)
groups_train = pd.read_parquet(Path(GROUPS_TRAIN_PATH)).squeeze()

cat_features = [col for col in X_train.columns if X_train[col].dtype in ['object', 'str']]

auc_results = []
for train_idx, val_idx in gkf.split(X_train, y_train, groups_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    eval_pool = Pool(X_val, y_val, cat_features=cat_features)

    model_cv = CatBoostClassifier(
        iterations=1500,
        learning_rate=0.1,
        depth=8,
        verbose=True,
        eval_metric='AUC',
        task_type='GPU',
        devices='0',
        early_stopping_rounds=20,
        auto_class_weights='Balanced',
    )

    model_cv.fit(X_tr, y_tr, cat_features=cat_features, eval_set=eval_pool, use_best_model=True)

    auc = model_cv.get_best_score()['validation']['AUC']
    auc_results.append(auc)
    print(f"Fold AUC: {auc:.4f}")

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7231485	best: 0.7231485 (0)	total: 201ms	remaining: 5m 1s
1:	total: 394ms	remaining: 4m 55s
2:	total: 583ms	remaining: 4m 51s
3:	total: 789ms	remaining: 4m 55s
4:	total: 965ms	remaining: 4m 48s
5:	test: 0.7359811	best: 0.7359811 (5)	total: 1.17s	remaining: 4m 51s
6:	total: 1.37s	remaining: 4m 52s
7:	total: 1.54s	remaining: 4m 47s
8:	total: 1.77s	remaining: 4m 53s
9:	total: 1.91s	remaining: 4m 44s
10:	test: 0.7410216	best: 0.7410216 (10)	total: 2.09s	remaining: 4m 42s
11:	total: 2.25s	remaining: 4m 38s
12:	total: 2.44s	remaining: 4m 39s
13:	total: 2.65s	remaining: 4m 41s
14:	total: 2.83s	remaining: 4m 39s
15:	test: 0.7436710	best: 0.7436710 (15)	total: 2.98s	remaining: 4m 36s
16:	total: 3.18s	remaining: 4m 37s
17:	total: 3.35s	remaining: 4m 35s
18:	total: 3.56s	remaining: 4m 37s
19:	total: 3.76s	remaining: 4m 37s
20:	test: 0.7456555	best: 0.7456555 (20)	total: 3.92s	remaining: 4m 36s
21:	total: 4.06s	remaining: 4m 32s
22:	total: 4.26s	remaining: 4m 33s
23:	total: 4.41s	remain

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7191432	best: 0.7191432 (0)	total: 274ms	remaining: 6m 51s
1:	total: 472ms	remaining: 5m 53s
2:	total: 660ms	remaining: 5m 29s
3:	total: 843ms	remaining: 5m 15s
4:	total: 1.03s	remaining: 5m 7s
5:	test: 0.7319674	best: 0.7319674 (5)	total: 1.26s	remaining: 5m 12s
6:	total: 1.46s	remaining: 5m 12s
7:	total: 1.66s	remaining: 5m 9s
8:	total: 1.86s	remaining: 5m 8s
9:	total: 2.01s	remaining: 4m 59s
10:	test: 0.7375375	best: 0.7375375 (10)	total: 2.24s	remaining: 5m 3s
11:	total: 2.41s	remaining: 4m 59s
12:	total: 2.63s	remaining: 5m
13:	total: 2.84s	remaining: 5m 1s
14:	total: 3s	remaining: 4m 57s
15:	test: 0.7402484	best: 0.7402484 (15)	total: 3.19s	remaining: 4m 55s
16:	total: 3.36s	remaining: 4m 53s
17:	total: 3.53s	remaining: 4m 50s
18:	total: 3.75s	remaining: 4m 52s
19:	total: 3.92s	remaining: 4m 49s
20:	test: 0.7424747	best: 0.7424747 (20)	total: 4.07s	remaining: 4m 46s
21:	total: 4.27s	remaining: 4m 47s
22:	total: 4.46s	remaining: 4m 46s
23:	total: 4.67s	remaining: 4m 46s

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7189147	best: 0.7189147 (0)	total: 248ms	remaining: 6m 12s
1:	total: 485ms	remaining: 6m 3s
2:	total: 724ms	remaining: 6m 1s
3:	total: 987ms	remaining: 6m 9s
4:	total: 1.21s	remaining: 6m 2s
5:	test: 0.7316370	best: 0.7316370 (5)	total: 1.47s	remaining: 6m 4s
6:	total: 1.74s	remaining: 6m 11s
7:	total: 1.99s	remaining: 6m 10s
8:	total: 2.18s	remaining: 6m 1s
9:	total: 2.35s	remaining: 5m 50s
10:	test: 0.7354790	best: 0.7354790 (10)	total: 2.78s	remaining: 6m 16s
11:	total: 3.1s	remaining: 6m 24s
12:	total: 3.28s	remaining: 6m 15s
13:	total: 3.46s	remaining: 6m 7s
14:	total: 3.66s	remaining: 6m 1s
15:	test: 0.7390985	best: 0.7390985 (15)	total: 3.85s	remaining: 5m 57s
16:	total: 4.1s	remaining: 5m 57s
17:	total: 4.25s	remaining: 5m 49s
18:	total: 4.54s	remaining: 5m 54s
19:	total: 4.84s	remaining: 5m 57s
20:	test: 0.7410836	best: 0.7410836 (20)	total: 5.09s	remaining: 5m 58s
21:	total: 5.28s	remaining: 5m 54s
22:	total: 5.46s	remaining: 5m 50s
23:	total: 5.69s	remaining: 5m 4

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7172056	best: 0.7172056 (0)	total: 203ms	remaining: 5m 3s
1:	total: 361ms	remaining: 4m 30s
2:	total: 549ms	remaining: 4m 34s
3:	total: 788ms	remaining: 4m 54s
4:	total: 1.05s	remaining: 5m 14s
5:	test: 0.7312417	best: 0.7312417 (5)	total: 1.27s	remaining: 5m 15s
6:	total: 1.48s	remaining: 5m 16s
7:	total: 1.72s	remaining: 5m 20s
8:	total: 1.93s	remaining: 5m 19s
9:	total: 2.14s	remaining: 5m 19s
10:	test: 0.7367007	best: 0.7367007 (10)	total: 2.29s	remaining: 5m 9s
11:	total: 2.44s	remaining: 5m 2s
12:	total: 2.7s	remaining: 5m 9s
13:	total: 2.91s	remaining: 5m 8s
14:	total: 3.1s	remaining: 5m 7s
15:	test: 0.7402819	best: 0.7402819 (15)	total: 3.28s	remaining: 5m 4s
16:	total: 3.42s	remaining: 4m 58s
17:	total: 3.69s	remaining: 5m 3s
18:	total: 3.87s	remaining: 5m 1s
19:	total: 4.11s	remaining: 5m 4s
20:	test: 0.7430904	best: 0.7430904 (20)	total: 4.37s	remaining: 5m 7s
21:	total: 4.64s	remaining: 5m 11s
22:	total: 4.82s	remaining: 5m 9s
23:	total: 5.04s	remaining: 5m 10s
2

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7165927	best: 0.7165927 (0)	total: 1.22s	remaining: 30m 34s
1:	total: 1.51s	remaining: 18m 50s
2:	total: 1.76s	remaining: 14m 39s
3:	total: 1.93s	remaining: 12m
4:	total: 2.15s	remaining: 10m 41s
5:	test: 0.7308991	best: 0.7308991 (5)	total: 2.57s	remaining: 10m 39s
6:	total: 2.79s	remaining: 9m 55s
7:	total: 2.96s	remaining: 9m 12s
8:	total: 3.23s	remaining: 8m 54s
9:	total: 3.57s	remaining: 8m 52s
10:	test: 0.7357080	best: 0.7357080 (10)	total: 3.83s	remaining: 8m 38s
11:	total: 4.01s	remaining: 8m 17s
12:	total: 4.25s	remaining: 8m 6s
13:	total: 4.55s	remaining: 8m 3s
14:	total: 4.76s	remaining: 7m 50s
15:	test: 0.7380576	best: 0.7380576 (15)	total: 4.91s	remaining: 7m 34s
16:	total: 5.13s	remaining: 7m 27s
17:	total: 5.36s	remaining: 7m 20s
18:	total: 5.57s	remaining: 7m 14s
19:	total: 5.78s	remaining: 7m 7s
20:	test: 0.7404129	best: 0.7404129 (20)	total: 5.95s	remaining: 6m 59s
21:	total: 6.28s	remaining: 7m 1s
22:	total: 6.53s	remaining: 6m 59s
23:	total: 6.78s	remaini

In [15]:
print(f'Baseline model CV AUC score mean: {np.array(auc_results).mean()}')
print(f'Baseline model CV STD: {np.array(auc_results).std()}')

Baseline model CV AUC score mean: 0.7551137864589691
Baseline model CV STD: 0.0013835038935334112


### Optuna

In [18]:
gss_sample = GroupShuffleSplit(n_splits=1, test_size=0.8, random_state=42)
sample_idx, _ = next(gss_sample.split(X_train, y_train, groups_train))

X_sample = X_train.iloc[sample_idx]
y_sample = y_train.iloc[sample_idx]
groups_sample = groups_train.iloc[sample_idx]

def objective(trial):
    params = {
        'iterations': 1500,
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.3, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 0, 1),
        'verbose': False,
        'eval_metric': 'AUC',
        'early_stopping_rounds': 20,
        'auto_class_weights': 'Balanced',
        'task_type': 'GPU',
        'devices': '0'
    }

    gss_cv = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=trial.number)
    train_idx, val_idx = next(gss_cv.split(X_sample, y_sample, groups_sample))

    X_tr, X_val = X_sample.iloc[train_idx], X_sample.iloc[val_idx]
    y_tr, y_val = y_sample.iloc[train_idx], y_sample.iloc[val_idx]

    cat_features = [col for col in X_sample.columns if X_sample[col].dtype in ['object', 'str']]

    model_opt = CatBoostClassifier(**params)
    eval_pool = Pool(X_val, y_val, cat_features=cat_features)
    model_opt.fit(X_tr, y_tr, cat_features=cat_features, eval_set=eval_pool, use_best_model=True)

    preds = model_opt.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, preds)

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"Best AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-05-10 23:34:55,455] A new study created in memory with name: no-name-23184a50-ee39-4da1-9d6e-6b917c55c571


  0%|          | 0/30 [00:00<?, ?it/s]

Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-05-10 23:35:14,004] Trial 0 finished with value: 0.747833738798715 and parameters: {'learning_rate': 0.09781801964118501, 'depth': 10, 'l2_leaf_reg': 7.587945476302646, 'bagging_temperature': 0.5986584841970366, 'random_strength': 0.15601864044243652}. Best is trial 0 with value: 0.747833738798715.


Default metric period is 5 because AUC is/are not implemented for GPU


...

[I 2026-05-10 23:45:54,421] Trial 27 finished with value: 0.7429723106000634 and parameters: {'learning_rate': 0.060381317771500784, 'depth': 6, 'l2_leaf_reg': 8.029773444502903, 'bagging_temperature': 0.4729006144801479, 'random_strength': 0.4215414060367655}. Best is trial 3 with value: 0.7540562124324601.


Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-05-10 23:46:23,338] Trial 28 finished with value: 0.7444270675190106 and parameters: {'learning_rate': 0.08617163549315635, 'depth': 9, 'l2_leaf_reg': 5.3350614103906695, 'bagging_temperature': 0.727537262678416, 'random_strength': 0.5173525652140668}. Best is trial 3 with value: 0.7540562124324601.


Default metric period is 5 because AUC is/are not implemented for GPU


[I 2026-05-10 23:46:41,490] Trial 29 finished with value: 0.7471762528353207 and parameters: {'learning_rate': 0.10420363964456503, 'depth': 10, 'l2_leaf_reg': 7.2658240146808115, 'bagging_temperature': 0.153343114999272, 'random_strength': 0.11315151444651206}. Best is trial 3 with value: 0.7540562124324601.
Best AUC: 0.7541
Best params: {'learning_rate': 0.06945227129221505, 'depth': 6, 'l2_leaf_reg': 5.72280788469014, 'bagging_temperature': 0.43194501864211576, 'random_strength': 0.2912291401980419}


### Entrenamiento final

In [7]:
cat_features = [col for col in X_train.columns if X_train[col].dtype in ['object', 'str']]

best_params = {
    'learning_rate': 0.06945227129221505,
    'depth': 6,
    'l2_leaf_reg': 5.72280788469014,
    'bagging_temperature': 0.43194501864211576,
    'random_strength': 0.2912291401980419
}

model_final = CatBoostClassifier(
    iterations=1500,
    **best_params,
    verbose=100,
    eval_metric='AUC',
    task_type='GPU',
    devices='0',
    early_stopping_rounds=20,
    auto_class_weights='Balanced',

)

model_final.fit(
    X_train, y_train,
    cat_features=cat_features,
    eval_set=Pool(X_test, y_test, cat_features=cat_features),
    use_best_model=True
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7059950	best: 0.7059950 (0)	total: 269ms	remaining: 6m 43s
100:	test: 0.7450317	best: 0.7450317 (100)	total: 17.1s	remaining: 3m 56s
200:	test: 0.7488944	best: 0.7488944 (200)	total: 34.4s	remaining: 3m 42s
300:	test: 0.7505404	best: 0.7505404 (300)	total: 54s	remaining: 3m 35s
400:	test: 0.7513635	best: 0.7513635 (400)	total: 1m 10s	remaining: 3m 14s
500:	test: 0.7519529	best: 0.7519529 (500)	total: 1m 28s	remaining: 2m 56s
600:	test: 0.7523313	best: 0.7523393 (598)	total: 1m 46s	remaining: 2m 39s
700:	test: 0.7526039	best: 0.7526041 (698)	total: 2m 4s	remaining: 2m 22s
bestTest = 0.7526980042
bestIteration = 760
Shrink model to first 761 iterations.


CatBoostClassifier(auto_class_weights='Balanced', bagging_temperature=0.43194501864211576, depth=6, devices='0', early_stopping_rounds=20, eval_metric='AUC', iterations=1500, l2_leaf_reg=5.72280788469014, learning_rate=0.06945227129221505, random_strength=0.2912291401980419, task_type='GPU', verbose=100)

In [8]:
pred_train_p = model_final.predict_proba(X_train)[:, 1]
pred_test_p = model_final.predict_proba(X_test)[:, 1]

pred_train = model_final.predict(X_train)
pred_test = model_final.predict(X_test)

In [9]:
print(f'AUC train: {roc_auc_score(y_train, pred_train_p)}')
print(f'AUC test: {roc_auc_score(y_test, pred_test_p)}')
print()
print(classification_report(y_test, pred_test ))
print(confusion_matrix(y_test, pred_test))

AUC train: 0.7669323509842775
AUC test: 0.7526980025675537

              precision    recall  f1-score   support

           0       0.90      0.66      0.76    102295
           1       0.34      0.71      0.46     25775

    accuracy                           0.67    128070
   macro avg       0.62      0.68      0.61    128070
weighted avg       0.79      0.67      0.70    128070

[[67047 35248]
 [ 7471 18304]]


In [10]:
model_final.save_model(str(Path(CATBOOST_MODEL_PATH)))